In [1]:
import pandas as pd

In [2]:
covid = pd.read_csv("../processed/Covid_19/metadata.csv")
pneumonia = pd.read_csv("../processed/Pneumonia/metadata.csv")
tb = pd.read_csv("../processed/TB/metadata.csv")
# pneumonia_bbox = pd.read_csv("../processed/all_data/pneumonia_bbox.csv")
# pneumonia_bbox = pneumonia_bbox[["file_name", "classification"]]

In [3]:
covid['file_name'] = covid['file_name'].str.replace('^NORMAL', 'Normal', regex=True)

In [4]:
# Create copies
pneumonia_df = pneumonia.copy()
covid_df = covid.copy()
# pneumonia_bbox_df = pneumonia_bbox.copy()
tb_df = tb.copy()

# Add bbox column where missing
pneumonia_df['bbox'] = 'none'
covid_df['bbox'] = 'none'

# Add missing width/height columns for pneumonia_bbox
# pneumonia_bbox_df['width'] = None
# pneumonia_bbox_df['height'] = None
# pneumonia_bbox_df['bbox'] = 'none'

# Add source column
pneumonia_df['source'] = 'Pneumonia Dataset'
covid_df['source'] = 'COVID Dataset'
# pneumonia_bbox_df['source'] = 'Pneumonia BBox Dataset'
tb_df['source'] = 'TB Dataset'

# Standardize column order
columns = ['file_name', 'width', 'height', 'classification', 'bbox', 'source']

pneumonia_df = pneumonia_df[columns]
covid_df = covid_df[columns]
# pneumonia_bbox_df = pneumonia_bbox_df[columns]
tb_df = tb_df[columns]

# Combine
combined_df = pd.concat(
    [pneumonia_df, covid_df, tb_df],
    ignore_index=True
)

print(combined_df.head())
all_metadata = combined_df

           file_name  width  height classification  bbox             source
0  IM-0001-0001.jpeg   1857    1317         Normal  none  Pneumonia Dataset
1  IM-0003-0001.jpeg   2111    1509         Normal  none  Pneumonia Dataset
2  IM-0005-0001.jpeg   2031    1837         Normal  none  Pneumonia Dataset
3  IM-0006-0001.jpeg   1663    1326         Normal  none  Pneumonia Dataset
4  IM-0007-0001.jpeg   2053    1818         Normal  none  Pneumonia Dataset


In [5]:
# Create a unique patient_id for each row based on classification
from collections import defaultdict

prefix_map = {
    'Tuberculosis': 'TB',
    'Pneumonia': 'PN',
    'COVID-19': 'CV',
    'Normal': 'NM'
}
counters = defaultdict(int)
patient_ids = []

for cls in all_metadata['classification']:
    prefix = prefix_map.get(cls, 'XX')
    counters[prefix] += 1
    patient_ids.append(f"{prefix}{counters[prefix]}")

all_metadata['patient_id'] = patient_ids
desired_order = ['file_name', 'classification', 'bbox', 'height', 'width', 'patient_id']
all_metadata = all_metadata[desired_order]
all_metadata.to_csv("../processed/all_data/metadata.csv", index=False)

In [6]:
all_metadata["classification"].value_counts()

classification
Normal          15575
Pneumonia        5618
COVID-19         3616
Tuberculosis     1211
Name: count, dtype: int64